# Vector Search Index Query

## Overview
This notebook demonstrates how to query a Databricks Vector Search index to retrieve relevant product documents using hybrid search (combining semantic similarity and keyword matching).

## Prerequisites
* Vector search endpoint: `product_docs_vector_search_endpoint`
* Vector search index: `agentic_ai.ecommerce.product_docs_vector_search_index`
* Environment variables configured:
  * `WORKSPACE_URL` - Databricks workspace URL
  * `SP_CLIENT_ID` - Service Principal client ID (optional)
  * `SP_CLIENT_SECRET` - Service Principal client secret (optional)

## What This Notebook Does
1. **Install Dependencies** - Installs the `databricks-vectorsearch` package
2. **Initialize Client** - Creates a VectorSearchClient to interact with the index
3. **Query Index** - Performs hybrid search to find relevant product documents

## Search Capabilities
* **Hybrid Search** - Combines semantic understanding with exact keyword matching
* **Configurable Results** - Specify number of results and which columns to return
* **Score-based Ranking** - Returns results ranked by relevance score

## Output Format
The search returns:
* `indexed_doc` - Full product document content
* `product_id` - Unique identifier for the product
* `score` - Relevance score (higher = more relevant)

In [0]:
# Install the Databricks Vector Search client library
# Note: databricks-vectorsearch has been renamed to databricks-ai-search
# but the old package name still works as a compatibility layer
%pip install databricks-vectorsearch

In [0]:
# Restart the Python kernel to load the newly installed package
# Note: All variables will be cleared after this restart
dbutils.library.restartPython()

In [0]:
# Import required libraries
import os
from databricks.vector_search.client import VectorSearchClient

# ========================================
# Configuration: Authentication Credentials
# ========================================
# Retrieve authentication details from environment variables
# These can be set in cluster configuration or notebook secrets
workspace_url = os.environ.get("WORKSPACE_URL")  # e.g., "https://your-workspace.cloud.databricks.com"
sp_client_id = os.environ.get("SP_CLIENT_ID")  # Service Principal client ID (optional)
sp_client_secret = os.environ.get("SP_CLIENT_SECRET")  # Service Principal secret (optional)

# ========================================
# Initialize Vector Search Client
# ========================================
# Create client to interact with Databricks Vector Search
# If service principal credentials are not provided, it will use notebook authentication
vsc = VectorSearchClient(
    workspace_url=workspace_url,
    service_principal_client_id=sp_client_id,
    service_principal_client_secret=sp_client_secret
)

# ========================================
# Get Reference to Vector Search Index
# ========================================
# Connect to the pre-created vector search index
# - endpoint_name: The vector search endpoint hosting the index
# - index_name: Fully qualified name (catalog.schema.table)
index = vsc.get_index(
    endpoint_name="product_docs_vector_search_endpoint",
    index_name="agentic_ai.ecommerce.product_docs_vector_search_index"
)

# ========================================
# Perform Hybrid Search Query
# ========================================
# Search the index using hybrid search (semantic + keyword matching)
# Parameters:
# - num_results: Number of top matches to return
# - columns: List of columns to include in the response
# - query_text: Natural language search query
# - query_type: "HYBRID" combines semantic similarity with keyword matching
#              (alternatives: "ANN" for pure semantic, "KEYWORD" for exact matching)
results = index.similarity_search(
    num_results=3,
    columns=["indexed_doc", "product_id"],
    query_text="Get the Arctic Guard details",
    query_type="HYBRID"
)

# Display the search results
results

## Usage Examples

### Different Query Types

```python
# Pure semantic search (vector similarity only)
results = index.similarity_search(
    num_results=5,
    columns=["indexed_doc", "product_id"],
    query_text="comfortable running shoes",
    query_type="ANN"  # Approximate Nearest Neighbor
)

# Keyword-only search (exact matching)
results = index.similarity_search(
    num_results=5,
    columns=["indexed_doc", "product_id"],
    query_text="Nike Air Max",
    query_type="KEYWORD"
)

# Hybrid search (recommended - best of both worlds)
results = index.similarity_search(
    num_results=5,
    columns=["indexed_doc", "product_id"],
    query_text="stylish comfortable athletic footwear",
    query_type="HYBRID"
)
```

### Filtering Results

```python
# Add filters to narrow down search results
results = index.similarity_search(
    num_results=3,
    columns=["indexed_doc", "product_id", "product_category"],
    query_text="winter jacket",
    query_type="HYBRID",
    filters={"product_category": "Fashion"}  # Only search within Fashion category
)
```

## Tips for Better Search Results

* **Use natural language** - The hybrid search understands context and semantics
* **Be specific** - Include relevant details in your query (e.g., "waterproof hiking boots" vs "boots")
* **Adjust num_results** - Start with 3-5 results, increase if needed
* **Choose the right query_type**:
  * `HYBRID` - Best for most use cases, balances semantic and keyword matching
  * `ANN` - When you want conceptually similar items (e.g., synonyms, related products)
  * `KEYWORD` - When you need exact matches (e.g., SKU, specific product names)

## Next Steps

* Integrate these search results into your application or chatbot
* Use the retrieved product documents for context in RAG (Retrieval Augmented Generation) applications
* Combine with filters to create category-specific search experiences